# 07 — Scenarios

What-if questions on the validated twin (v4).

**Scenario: add one LITHOGRAPHY machine.** Which one? A copy of a machine that runs heavily loaded
recipes (much work, few machines), or a copy of one that runs recipes many machines can already run?

In [9]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd

from wafer_twin.data import load_tables, build_process
from wafer_twin.simulation import (
    load_twin_inputs, load_priority, load_same_recipe_factor, load_load_effect, run_twin,
)

tables = load_tables()
process = build_process(tables)
real = process[process["WORKSHOP"] == "LITHOGRAPHY"]

arrivals, machine_recipes, gamma, _ = load_twin_inputs(tables, process)
prod, rec = load_priority("v3")

# twin v4 settings
twin_kwargs = dict(
    recipe_scores=rec,
    same_recipe_factor=load_same_recipe_factor("v3"),
    load_effect=load_load_effect(),
)


def replicate(machines, seeds=range(5)):
    rows = []
    for s in seeds:
        out = run_twin(arrivals, machines, gamma, prod, seed=s, **twin_kwargs)
        w = out.groupby("PRODUCT")["WAIT_MIN"].mean()
        w["ALL"] = out["WAIT_MIN"].mean()
        rows.append(w.rename(f"seed_{s}"))
    return pd.DataFrame(rows)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# Recipe pressure and machine load

n_machines = pd.Series({r: sum(r in recs for recs in machine_recipes.values()) for r in gamma.index})
work_h = real.groupby("RECIPE")["PROCESS_TIME_MIN"].sum() / 60

# hours of work each able machine would have to absorb over 2 years
pressure = work_h / n_machines

used = {m: [r for r in recs if r in pressure.index] for m, recs in machine_recipes.items()}

machines = pd.DataFrame({
    "n_used_recipes": {m: len(r) for m, r in used.items()},
    "n_unused_recipes": {m: len(machine_recipes[m]) - len(r) for m, r in used.items()},
    "mean_pressure_h": {m: pressure[r].mean() for m, r in used.items()},
    "runs_0027": {m: "RECIPE_0027" in r for m, r in used.items()},
    "runs_0021": {m: "RECIPE_0021" in r for m, r in used.items()},
}).sort_values("mean_pressure_h", ascending=False)

loaded = machines["mean_pressure_h"].idxmax()
light = machines["mean_pressure_h"].idxmin()
print("most loaded recipes:", loaded, used[loaded])
print("least loaded recipes:", light, used[light])
machines.round(0)

most loaded recipes: MACHINE_0016 ['RECIPE_0012', 'RECIPE_0027', 'RECIPE_0028']
least loaded recipes: MACHINE_0009 ['RECIPE_0010', 'RECIPE_0011', 'RECIPE_0017', 'RECIPE_0026']


,n_used_recipes,n_unused_recipes,mean_pressure_h,runs_0027,runs_0021
MACHINE_0016,3,0,7233.0,True,False
MACHINE_0026,3,0,5515.0,False,True
MACHINE_0010,8,1,5260.0,True,True
MACHINE_0025,4,1,4606.0,False,False
MACHINE_0002,6,1,4565.0,False,False
MACHINE_0032,5,0,4448.0,False,True
MACHINE_0035,4,0,4268.0,True,False
MACHINE_0034,5,1,4028.0,False,False
MACHINE_0036,5,0,4010.0,False,False
MACHINE_0007,9,1,3773.0,False,False


In [6]:
# run the scenario

def with_copy_of(src):
    m = dict(machine_recipes)
    m[f"NEW_{src}"] = machine_recipes[src]  # same recipes as the machine we copy
    return m


t = time.perf_counter()
rep_base = replicate(machine_recipes)
rep_loaded = replicate(with_copy_of(loaded))
rep_light = replicate(with_copy_of(light))
print(f"15 runs in {time.perf_counter() - t:.0f} s")

scen = pd.DataFrame({
    "real": real.groupby("PRODUCT")["WAIT_MIN"].mean(),
    "baseline": rep_base.mean(),
    f"+ copy of {loaded}": rep_loaded.mean(),
    f"+ copy of {light}": rep_light.mean(),
})
scen.loc["ALL", "real"] = real["WAIT_MIN"].mean()
scen.round(1)

15 runs in 246 s


,real,baseline,+ copy of MACHINE_0016,+ copy of MACHINE_0009
PRODUCT,,,,
ALL,155.7,164.4,80.6,96.6
PRODUCT_0001,109.6,113.9,59.1,71.9
PRODUCT_0002,292.0,312.5,140.9,166.6
PRODUCT_0003,132.5,135.1,70.9,83.3
PRODUCT_0004,159.9,177.1,83.2,102.5


In [7]:
# Gain and spread

spread = pd.DataFrame({
    name: [r["ALL"].mean(), r["ALL"].min(), r["ALL"].max()]
    for name, r in [("baseline", rep_base), (f"+ copy of {loaded}", rep_loaded), (f"+ copy of {light}", rep_light)]
}, index=["mean", "min", "max"]).T

spread["gain_vs_baseline"] = 1 - spread["mean"] / spread.loc["baseline", "mean"]
spread.round(3)

,mean,min,max,gain_vs_baseline
baseline,164.431,158.303,171.586,0.000
+ copy of MACHINE_0016,80.604,78.774,81.775,0.510
+ copy of MACHINE_0009,96.556,95.638,98.569,0.413


### Takeaways :

- MACHINE_0016 runs only heavily loaded recipes (0012, 0027, 0028; mean pressure 7,233 h),
  MACHINE_0009 only recipes many machines can already run (0010, 0011, 0017, 0026; 1,255 h)
- One extra machine (+2.5% capacity) roughly halves waiting: 164 → 81 min with a copy of MACHINE_0016 (−51%),
  164 → 97 min with a copy of MACHINE_0009 (−41%). At 99% utilization a little spare capacity changes everything
- Where the capacity goes matters: copying the loaded machine gains 10 points more, with non-overlapping ranges
  (78.8–81.8 vs 95.6–98.6)
- The "easy" machine still helps a lot: it takes easy recipes off other machines, which then have more time
  for loaded ones. Capacity spreads through the machines' shared recipes
- PRODUCT_0002, the least prioritised, gains most: 313 → 141 min per operation with a copy of MACHINE_0016
- Less saturated, the workshop is also more stable: ±2% between seeds vs ±4% today
- Limits: the twin overestimates waiting by ~6%, so relative gains are more reliable than minutes;
  the rest of the fab doesn't react (arrivals are the real ones); the load effect (slower when less busy)
  is already included

## Scenario 2: same priority for every product

Today PRODUCT_0002 waits ~2.7× more than PRODUCT_0001. What if all products had the same score?
Recipe scores, the same-recipe effect and the load effect are kept: they come from technical constraints
(recipe load, mask changes), not from a management choice.

Expected from the toy example (notebook 05): priority redistributes waiting between products
rather than reducing the total.

In [11]:
def replicate(machines, scores=None, seeds=range(5)):
    scores = scores or prod  # default: fitted product scores
    rows = []
    for s in seeds:
        out = run_twin(arrivals, machines, gamma, scores, seed=s, **twin_kwargs)
        w = out.groupby("PRODUCT")["WAIT_MIN"].mean()
        w["ALL"] = out["WAIT_MIN"].mean()
        rows.append(w.rename(f"seed_{s}"))
    return pd.DataFrame(rows)

In [12]:
# run with equal product scores
equal = {p: 1.0 for p in prod}

t = time.perf_counter()
rep_equal = replicate(machine_recipes, scores=equal)
print(f"5 runs in {time.perf_counter() - t:.0f} s")

prio = pd.DataFrame({
    "baseline": rep_base.mean(),
    "equal_scores": rep_equal.mean(),
})
prio["change"] = prio["equal_scores"] / prio["baseline"] - 1
prio.round(3)

5 runs in 117 s


,baseline,equal_scores,change
PRODUCT,,,
PRODUCT_0001,113.910,177.172,0.555
PRODUCT_0002,312.460,156.406,-0.499
PRODUCT_0003,135.136,159.836,0.183
PRODUCT_0004,177.104,151.722,-0.143
ALL,164.431,162.909,-0.009


In [13]:
# Spread between products and between seeds

def product_gap(rep):
    # how much longer the worst-off product waits than the best-off one
    means = rep.drop(columns="ALL").mean()
    return means.max() / means.min()


print(f"worst / best product, baseline:     {product_gap(rep_base):.2f}")
print(f"worst / best product, equal scores: {product_gap(rep_equal):.2f}")
print()
pd.DataFrame({
    name: [r["ALL"].mean(), r["ALL"].min(), r["ALL"].max()]
    for name, r in [("baseline", rep_base), ("equal_scores", rep_equal)]
}, index=["mean", "min", "max"]).T.round(1)

worst / best product, baseline:     2.74
worst / best product, equal scores: 1.17



,mean,min,max
baseline,164.4,158.3,171.6
equal_scores,162.9,159.9,166.0


### Takeaways :

- Total waiting does not change: 164.4 → 162.9 min (−0.9%), overlapping ranges (158–172 vs 160–166).
  Removing product priority neither speeds up nor slows down the workshop
- Waiting is redistributed: PRODUCT_0002 312 → 156 min (−50%), PRODUCT_0004 −14%,
  PRODUCT_0003 +18%, PRODUCT_0001 114 → 177 min (+56%)
- Inequality almost disappears: worst/best product ratio 2.74 → 1.17
- Without priority, PRODUCT_0001 becomes the worst-off: the remaining gaps come from each product's recipes.
  Its current priority may partly offset this (hypothesis)
- The twin doesn't say which rule is right (that depends on product value and due dates),
  but it prices the trade-off: −156 min per operation for PRODUCT_0002, +63 min for PRODUCT_0001,
  same total
- Consistent with the toy example (notebook 05), now in a realistic workshop